# BPS residual report — 2016/17, 2017/18, 2018/19

**This notebook is a measurement, not a model.** It is deliberately a notebook rather than a
module so it cannot be imported and cannot quietly become a dependency of anything downstream
(plan §6.6, task 20).

Finding 2 established that bonus (BPS) is not exactly derivable in any season, but that
2016/17–2018/19 publish most of the BPS input table alongside FPL's own published `bps` total.
That turns "model bonus blind from proxies" into "model bonus against a measured residual".
This notebook computes that residual using every BPS term we can observe from
`facts/player_fixture`, exactly as scored per the official rules in
`docs/Fantasy Premier League Scoring.md`, and reports:

1. the distribution of `bps_observed - bps_fpl`, overall and by position;
2. which terms are missing and therefore plausibly explain the residual;
3. how often the residual changes the top-3 bonus ordering within a match — the only thing
   that actually matters, since bonus is awarded on rank, not on level.

No model is fitted here. No new dependency is introduced (plain `polars`, already a project
dependency). No production code path reads this notebook's output.

In [1]:
import polars as pl

SEASONS = ["2016-17", "2017-18", "2018-19"]


def load(season: str) -> pl.DataFrame:
    return pl.read_parquet(f"../data/facts/player_fixture/season={season}/part.parquet")


df = pl.concat([load(s) for s in SEASONS])
print("rows:", df.height)

rows: 67936


## Observed BPS formula

Every term below is taken verbatim from `docs/Fantasy Premier League Scoring.md`'s BPS table
and computed only from columns Finding 2 confirmed are populated for these three seasons.

**Terms we can compute:** minutes bands, goals by position, assists, clean sheets (GK/DEF),
saves, penalty saves, CBI (per 3), recoveries (per 3), chances created, big chances created,
open play crosses, successful tackles, dribbles, winning goals, pass-completion bands
(≥30 attempts), goals conceded (GK/DEF), penalties conceded/missed, cards, own goals, big
chances missed, errors leading to a goal/attempt, offside, fouls conceded.

**Terms we cannot compute — not published for these seasons (Finding 2):** shots on target,
saves inside the box specifically, saves from a big chance specifically, goalline clearances,
fouls *won* (as opposed to conceded, which we do have). We also cannot split penalty goals from
open-play goals, so all goals are scored at the non-penalty rate — a second, smaller source of
bias beyond the five missing terms above.

**Caveat (plan Risk R1):** there is no overlap season between 2018/19 and 2025/26, so this
cannot prove Opta's definitions of these fields held constant across the archive's own history.
It measures reproducibility from FPL's *own* published inputs to FPL's *own* published `bps`,
which is a lower bar than proving the terms are what we think they are — but it is the honest
bar available with zero external data.

In [2]:
def observed_bps(frame: pl.DataFrame) -> pl.DataFrame:
    pass_pct = pl.when(pl.col("attempted_passes") >= 30).then(
        pl.col("completed_passes") / pl.col("attempted_passes") * 100
    ).otherwise(None)
    pass_band = (
        pl.when(pass_pct >= 90).then(6)
        .when(pass_pct >= 80).then(4)
        .when(pass_pct >= 70).then(2)
        .otherwise(0)
    )
    minutes_bps = (
        pl.when(pl.col("minutes") >= 60).then(6).when(pl.col("minutes") >= 1).then(3).otherwise(0)
    )
    goal_bps = (
        pl.when(pl.col("position").is_in(["GK", "DEF"])).then(pl.col("goals_scored") * 12)
        .when(pl.col("position") == "MID").then(pl.col("goals_scored") * 18)
        .otherwise(pl.col("goals_scored") * 24)
    )
    clean_sheet_bps = pl.when(
        (pl.col("position").is_in(["GK", "DEF"]))
        & (pl.col("minutes") >= 60)
        & (pl.col("goals_conceded") == 0)
    ).then(12).otherwise(0)
    conceded_bps = pl.when(pl.col("position").is_in(["GK", "DEF"])).then(
        -4 * pl.col("goals_conceded")
    ).otherwise(0)

    return frame.with_columns(
        (
            minutes_bps
            + goal_bps
            + pl.col("assists") * 9
            + clean_sheet_bps
            + pl.col("saves") * 2
            + pl.col("penalties_saved") * 7
            + (pl.col("cbi") / 3).floor() * 1
            + (pl.col("recoveries") / 3).floor() * 1
            + pl.col("key_passes") * 1
            + pl.col("big_chances_created") * 3
            + pl.col("open_play_crosses") * 1
            + pl.col("tackles") * 2
            + pl.col("dribbles") * 1
            + pl.col("winning_goals") * 3
            + pass_band
            + conceded_bps
            + pl.col("penalties_conceded") * -3
            + pl.col("penalties_missed") * -6
            + pl.col("yellow_cards") * -3
            + pl.col("red_cards") * -9
            + pl.col("own_goals") * -6
            + pl.col("big_chances_missed") * -3
            + pl.col("errors_leading_to_goal") * -3
            + pl.col("errors_leading_to_goal_attempt") * -1
            + pl.col("offside") * -1
            + pl.col("fouls") * -1
        ).alias("bps_observed")
    )


df2 = observed_bps(df).with_columns(
    (pl.col("bps_observed") - pl.col("bps_fpl")).alias("residual")
)

## Residual distribution, overall and by position

In [3]:
with pl.Config(tbl_cols=-1, tbl_rows=-1):
    print(df2.select(["bps_observed", "bps_fpl", "residual"]).describe())

shape: (9, 4)
┌────────────┬──────────────┬───────────┬─────────┐
│ statistic  ┆ bps_observed ┆ bps_fpl   ┆ residual  │
│ ---        ┆ ---          ┆ ---       ┆ ---       │
│ str        ┆ f64          ┆ f64       ┆ f64       │
╞════════════╪══════════════╪═══════════╪═════════╡
│ count      ┆ 67936.0      ┆ 67936.0   ┆ 67936.0   │
│ null_count ┆ 0.0          ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 5.798987     ┆ 6.283399  ┆ -0.484412 │
│ std        ┆ 10.138201    ┆ 10.002017 ┆ 3.521939  │
│ min        ┆ -25.0        ┆ -19.0     ┆ -30.0     │
│ 25%        ┆ 0.0          ┆ 0.0       ┆ 0.0       │
│ 50%        ┆ 0.0          ┆ 0.0       ┆ 0.0       │
│ 75%        ┆ 9.0          ┆ 11.0      ┆ 0.0       │
│ max        ┆ 121.0        ┆ 114.0     ┆ 13.0      │
└────────────┴──────────────┴───────────┴─────────┘


In [4]:
with pl.Config(tbl_cols=-1, tbl_rows=-1):
    print(
        df2.group_by("position").agg(
            pl.col("residual").mean().alias("mean_residual"),
            pl.col("residual").std().alias("std_residual"),
            pl.col("residual").abs().mean().alias("mean_abs_residual"),
            pl.len().alias("n"),
        ).sort("position")
    )

shape: (4, 5)
┌──────────┬─────────────┬────────────┬──────────────────┬──────┐
│ position ┆ mean_residual ┆ std_residual ┆ mean_abs_residual ┆ n     │
│ ---      ┆ ---           ┆ ---          ┆ ---               ┆ ---   │
│ str      ┆ f64           ┆ f64          ┆ f64               ┆ u32   │
╞══════════╪═════════════╪════════════╪══════════════════╪══════╡
│ DEF      ┆ -2.403708     ┆ 4.363303     ┆ 2.524552          ┆ 22707 │
│ FWD      ┆ 1.070062      ┆ 1.892278     ┆ 1.084155          ┆ 9934  │
│ GK       ┆ -1.904322     ┆ 4.066184     ┆ 1.904606          ┆ 7034  │
│ MID      ┆ 0.86469       ┆ 1.736994     ┆ 0.946569          ┆ 28261 │
└──────────┴─────────────┴────────────┴──────────────────┴──────┘


**Reading:** the median residual is 0 for every group (most rows score 0 BPS both ways —
unused substitutes), but defenders and goalkeepers are the most affected by the missing terms:
on average we under-count DEF by 2.4 BPS and GK by 1.9 BPS, plausibly because defenders and
keepers accrue more of the missing defensive terms (goalline clearances) and because our
clean-sheet/conceded terms are all-or-nothing while save-quality terms (saves in the box, saves
from a big chance) are entirely missing for goalkeepers. Forwards and midfielders are
over-counted on average by ~1 BPS, consistent with missing negative terms (fouls won isn't
missing as a negative, but shot-on-target is a positive term we cannot add for attacking
players, so the sign of the remaining bias is less clear-cut than for defenders).

## Does the residual change who gets bonus points?

Bonus is awarded on **rank within a match**, not on the BPS level itself. The residual only
matters in practice if it changes which three players are the top scorers. This compares the
top-3 BPS scorers (by player, restricted to those who actually played) under our observed
formula against FPL's own published `bps`, per fixture.

In [5]:
df3 = df2.filter(pl.col("minutes") > 0)


def top3(frame: pl.DataFrame, col: str) -> pl.DataFrame:
    return (
        frame.sort(["season", "fixture_id", col], descending=[False, False, True])
        .group_by(["season", "fixture_id"], maintain_order=True)
        .head(3)
        .select("season", "fixture_id", "player_id")
        .with_columns(pl.lit(1).alias("in_top3"))
    )


fpl_top3 = top3(df3, "bps_fpl")
obs_top3 = top3(df3, "bps_observed")

fixtures = df3.select("season", "fixture_id").unique().height
merged = fpl_top3.join(
    obs_top3, on=["season", "fixture_id", "player_id"], how="full", suffix="_obs"
)
disagreements = merged.filter(pl.col("in_top3").is_null() | pl.col("in_top3_obs").is_null())
n_disagree = disagreements.select("season", "fixture_id").unique().height

print("total fixtures:", fixtures)
print("fixtures where observed top-3 differs from FPL top-3 membership:", n_disagree)
print("share:", round(n_disagree / fixtures, 4))

total fixtures: 1140
fixtures where observed top-3 differs from FPL top-3 membership: 538
share: 0.4719


## Conclusion

The five terms missing from 2016/17–2018/19 (shots on target, saves inside the box, saves from
a big chance, goalline clearances, fouls won) are **not a rounding error**: our observed
reconstruction changes the top-3 bonus-scoring membership in **47% of fixtures** (538 of 1,140)
relative to FPL's own published `bps`. Combined with a systematic per-position bias (defenders
and goalkeepers under-counted by ~2–2.5 BPS on average, attackers over-counted by ~1 BPS), this
confirms Finding 2's practical conclusion: **bonus must be modelled, or observed as FPL's
published value, never derived from these proxies and treated as ground truth.** The BPS input
columns remain useful as *features* for a bonus model — they carry real, position-differentiated
signal — but a model trained to reproduce `bps_fpl` from them, not a hand-written formula
reproducing the rules, is the only approach worth pursuing (phase 8+).

This finding, including the R1 definition-drift caveat, is recorded in the design spec
(`docs/superpowers/specs/2026-07-30-fpl-data-layer-design.md`, §4) alongside the other phase 6
findings.